# Multimodal Fraud Detection - Comprehensive Pipeline

This notebook demonstrates the complete end-to-end pipeline for multimodal fraud detection on GPU-accelerated system.

## Pipeline Flow:
```
Online Payment Fraud Dataset (/home/ec2-user/csvdata)
          ↓
    Tabular Data
          ↓
FraudDataPreprocessor
          ↓
          ├─────────────────────┐
          ↓                     ↓
QR Code Images (/home/ec2-user/qrdata)  |
          ↓                     |
     Image Data                 |
          ↓                     |
  QRCodePreprocessor            |
          ↓                     |
          └─────────────────────┤
                                ↓
         MultimodalDataPreprocessor
                                ↓
         Processed Data Storage
                                ↓
          ├─────────────────────┐
          ↓                     ↓
TabularTransformerEncoder  VisionTransformerEncoder
          ↓                     ↓
          └─────────┬───────────┘
                    ↓
         Cross-Modal Fusion
                    ↓
           Fraud Detection
                    ↓
    Model Saved to /home/ec2-user/model
```

## GPU-Accelerated Setup:
- **GPU**: NVIDIA A10G (24GB VRAM)
- **Data Directories**:
  - QR Codes: `/home/ec2-user/qrdata`
  - CSV Data: `/home/ec2-user/csvdata`
  - Models: `/home/ec2-user/model`

## Datasets Used:
1. **Online Payments Fraud Detection Dataset**: Transaction data with fraud labels
2. **Benign and Malicious QR Codes Dataset**: QR code images for visual analysis

## 1. Setup and Verify Environment

Verify that the system is properly configured with GPU access and required directories.

In [ ]:
import os
import sys
import tensorflow as tf
import numpy as np
import pandas as pd

print("="*70)
print("Environment Check")
print("="*70)

# Check Python version
print(f"\nPython version: {sys.version}")

# Check TensorFlow and GPU
print(f"\nTensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU Available: {len(gpus) > 0}")
if gpus:
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu.name}")
        # Set memory growth to avoid allocating all GPU memory at once
        tf.config.experimental.set_memory_growth(gpu, True)
    print("\n✓ GPU memory growth enabled")
else:
    print("\n⚠ No GPU detected. Training will use CPU (slower).")

# Enable mixed precision for faster training on GPU
if gpus:
    from tensorflow.keras import mixed_precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("✓ Mixed precision enabled for faster GPU training")

# Verify EC2 directories
print("\n" + "="*70)
print("Directory Check")
print("="*70)

ec2_dirs = {
    'QR Data': '/home/ec2-user/qrimages/QR codes',
    'CSV Data': '/home/ec2-user/csvdata',
    'Models': '/home/ec2-user/model',
    'Logs': '/home/ec2-user/model/logs'
}

all_exist = True
for name, path in ec2_dirs.items():
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {name}: {path}")
    if not exists:
        all_exist = False

if not all_exist:
    print("\n⚠ Some directories are missing. Run setup script first:")
    print("  bash setup_ec2_jupyter.sh")
else:
    print("\n✓ All required directories exist")

## 2. Add Repository to Python Path

Ensure the repository code is accessible.

In [ ]:
# Add repository to Python path (adjust if your repo is in a different location)
import sys
import os

# Assuming the repository is cloned in ec2-user's home directory
REPO_PATH = '/home/ec2-user/test'

if os.path.exists(REPO_PATH):
    sys.path.insert(0, REPO_PATH)
    print(f"✓ Repository path added: {REPO_PATH}")
else:
    print(f"⚠ Repository not found at {REPO_PATH}")
    print("\nClone the repository first:")
    print("  cd /home/ec2-user")
    print("  git clone https://github.com/go2nishantnig/test.git")

## 3. Import Required Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import project modules with EC2 configuration
from config.config import (
    MODEL_CONFIG, 
    TRAINING_CONFIG, 
    QRCODE_DATASET_PATH,
    DATA_DIR,
    MODEL_SAVE_DIR,
    FRAUD_DATASET_PATH,
    QRCODE_DATASET_PATH
)

from src.models.transformer_model import (
    MultimodalFraudDetectionTransformer
)

from src.utils.data_preprocessing import (
    FraudDataPreprocessor,
    QRCodePreprocessor,
    MultimodalDataPreprocessor
)

print("✓ All libraries imported successfully")
print(f"\nConfiguration:")
print(f"  - Model will be saved to: {MODEL_SAVE_DIR}")
print(f"  - CSV data location: {DATA_DIR}")
print(f"  - QR data location: {QRCODE_DATASET_PATH}")

## 4. Load and Preprocess Data

### 4.1 Tabular Data Preprocessing

In [ ]:
print("="*70)
print("Tabular Data Preprocessing (Online Payment Fraud Dataset)")
print("="*70)

# Initialize tabular data preprocessor
fraud_preprocessor = FraudDataPreprocessor()

# Try to load real data from CSV, otherwise generate synthetic data
csv_file = os.path.join(DATA_DIR, 'online_payments_fraud.csv')

if os.path.exists(csv_file):
    print(f"\n✓ Found real dataset at: {csv_file}")
    df = pd.read_csv(csv_file)
    print(f"  Loaded {len(df)} transactions")
    
    # Preprocess real data
    X_tabular, y = fraud_preprocessor.preprocess_data(df)
    print(f"\n✓ Preprocessed real data")
else:
    print(f"\n⚠ Real dataset not found at: {csv_file}")
    print("  Generating synthetic data for demonstration...")
    
    # Generate synthetic data
    X_tabular, y = fraud_preprocessor.generate_synthetic_data(n_samples=5000, fraud_ratio=0.1)
    print(f"\n✓ Generated {len(y)} synthetic transactions")

print(f"\nTabular Data Summary:")
print(f"  Shape: {X_tabular.shape}")
print(f"  Fraud ratio: {np.mean(y):.2%}")
print(f"  Features: {X_tabular.shape[1]}")

# Visualize fraud distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
fraud_counts = pd.Series(y.flatten()).value_counts()
plt.bar(['Normal', 'Fraud'], fraud_counts.values)
plt.title('Transaction Distribution')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.pie(fraud_counts.values, labels=['Normal', 'Fraud'], autopct='%1.1f%%')
plt.title('Fraud Ratio')
plt.tight_layout()
plt.show()

print("\n✓ Tabular data preprocessing complete")

### 4.2 Image Data Preprocessing (QR Codes)

In [ ]:
print("="*70)
print("Image Data Preprocessing (QR Code Images)")
print("="*70)

# Initialize QR code preprocessor
image_size = MODEL_CONFIG.get('image_size', (128, 128))
qr_preprocessor = QRCodePreprocessor(image_size=image_size)

# Try to load real QR code images, otherwise generate synthetic patterns
qr_benign_path = os.path.join(QRCODE_DATASET_PATH, 'benign', 'benign')
qr_malicious_path = os.path.join(QRCODE_DATASET_PATH, 'malicious', 'malicious')

if os.path.exists(qr_benign_path) and os.path.exists(qr_malicious_path):
    print(f"\n✓ Found real QR code images in: {QRCODE_DATASET_PATH}")
    
    # Load real images
    benign_images = qr_preprocessor.load_images(qr_benign_path, max_images=2500)
    malicious_images = qr_preprocessor.load_images(qr_malicious_path, max_images=2500)
    
    print(f"  Loaded {len(benign_images)} benign QR codes")
    print(f"  Loaded {len(malicious_images)} malicious QR codes")
    
    # Combine and create labels
    X_images = np.vstack([benign_images, malicious_images])
    y_images = np.vstack([
        np.zeros((len(benign_images), 1)),
        np.ones((len(malicious_images), 1))
    ])
    
    # Ensure we have the same number of samples as tabular data
    if len(X_images) != len(X_tabular):
        print(f"\n  Adjusting image count to match tabular data ({len(X_tabular)} samples)...")
        if len(X_images) > len(X_tabular):
            X_images = X_images[:len(X_tabular)]
            y_images = y_images[:len(X_tabular)]
        else:
            # Need more images - generate synthetic ones
            additional_needed = len(X_tabular) - len(X_images)
            additional_images = qr_preprocessor.generate_synthetic_images(
                n_samples=additional_needed
            )
            additional_labels = np.random.randint(0, 2, (additional_needed, 1))
            X_images = np.vstack([X_images, additional_images])
            y_images = np.vstack([y_images, additional_labels])
    
    print(f"\n✓ Preprocessed real QR code images")
else:
    print(f"\n⚠ Real QR code images not found in: {QRCODE_DATASET_PATH}")
    print("  Expected structure: benign/benign/ and malicious/malicious/ subdirectories")
    print("  Generating synthetic QR code patterns for demonstration...")
    
    # Generate synthetic QR code patterns
    X_images = qr_preprocessor.generate_synthetic_images(n_samples=len(X_tabular))
    y_images = np.random.randint(0, 2, (len(X_tabular), 1))
    print(f"\n✓ Generated {len(X_images)} synthetic QR code patterns")

print(f"\nImage Data Summary:")
print(f"  Shape: {X_images.shape}")
print(f"  Image size: {image_size}")
print(f"  Channels: RGB")
print(f"  Normalized range: [0, 1]")

# Visualize sample QR codes
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_images[i])
    label = 'Malicious' if y_images[i] == 1 else 'Benign'
    ax.set_title(f'Sample {i+1}: {label}')
    ax.axis('off')
plt.suptitle('Sample QR Code Images')
plt.tight_layout()
plt.show()

print("\n✓ Image data preprocessing complete")

### 4.3 Combine Multimodal Data

In [ ]:
print("="*70)
print("Multimodal Data Integration")
print("="*70)

# Initialize multimodal preprocessor
multimodal_preprocessor = MultimodalDataPreprocessor(image_size=image_size)
multimodal_preprocessor.fraud_preprocessor = fraud_preprocessor
multimodal_preprocessor.qr_preprocessor = qr_preprocessor

# Use the labels from tabular data (more reliable)
from sklearn.model_selection import train_test_split

# Split data
indices = np.arange(len(X_tabular))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TRAINING_CONFIG['validation_split'],
    stratify=y,
    random_state=42
)

# Create train/test splits
X_train_tabular = X_tabular[train_idx]
X_test_tabular = X_tabular[test_idx]
X_train_images = X_images[train_idx]
X_test_images = X_images[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print(f"\nData Split:")
print(f"  Training samples: {len(y_train)}")
print(f"  Testing samples: {len(y_test)}")
print(f"  Training fraud ratio: {np.mean(y_train):.2%}")
print(f"  Testing fraud ratio: {np.mean(y_test):.2%}")

print(f"\nTraining Data Shapes:")
print(f"  Tabular: {X_train_tabular.shape}")
print(f"  Images: {X_train_images.shape}")
print(f"  Labels: {y_train.shape}")

print("\n✓ Multimodal data integration complete")

## 5. Build Multimodal Transformer Model

In [ ]:
print("="*70)
print("Building Multimodal Transformer Model")
print("="*70)

# Create model
multimodal_model = MultimodalFraudDetectionTransformer(MODEL_CONFIG)
model = multimodal_model.compile_model(learning_rate=TRAINING_CONFIG['learning_rate'])

print("\n✓ Model compiled successfully")
print("\nModel Architecture:")
model.summary()

# Count parameters
trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_weights])
print(f"\nTotal trainable parameters: {trainable_params:,}")

## 6. Train the Model

In [ ]:
print("="*70)
print("Training Multimodal Fraud Detection Model")
print("="*70)

# Setup callbacks
from tensorflow import keras

# Model checkpoint - save best model
checkpoint_path = os.path.join(MODEL_SAVE_DIR, 'multimodal_fraud_detection_best.keras')
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Early stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=TRAINING_CONFIG['early_stopping_patience'],
    restore_best_weights=True,
    verbose=1
)

# Learning rate reduction
lr_reduction = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# TensorBoard
log_dir = os.path.join(MODEL_SAVE_DIR, 'logs', datetime.now().strftime('%Y%m%d-%H%M%S'))
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

print(f"\nCallbacks configured:")
print(f"  ✓ Model checkpoint: {checkpoint_path}")
print(f"  ✓ Early stopping (patience={TRAINING_CONFIG['early_stopping_patience']})")
print(f"  ✓ Learning rate reduction")
print(f"  ✓ TensorBoard logs: {log_dir}")

# Train model
print(f"\nStarting training...")
print(f"  Epochs: {TRAINING_CONFIG['epochs']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")

history = model.fit(
    [X_train_tabular, X_train_images],
    y_train,
    validation_data=([X_test_tabular, X_test_images], y_test),
    epochs=TRAINING_CONFIG['epochs'],
    batch_size=TRAINING_CONFIG['batch_size'],
    callbacks=[
        checkpoint_callback,
        early_stopping,
        lr_reduction,
        tensorboard_callback
    ],
    verbose=1
)

print("\n✓ Training complete!")

## 7. Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

# Learning rate
if 'lr' in history.history:
    axes[2].plot(history.history['lr'])
    axes[2].set_title('Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_yscale('log')
    axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'training_history.png'), dpi=150)
plt.show()

print(f"\n✓ Training history plot saved to: {MODEL_SAVE_DIR}/training_history.png")

## 8. Evaluate Model Performance

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

print("="*70)
print("Model Evaluation")
print("="*70)

# Make predictions
y_pred_proba = model.predict([X_test_tabular, X_test_images])
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate metrics
test_loss, test_accuracy = model.evaluate(
    [X_test_tabular, X_test_images],
    y_test,
    verbose=0
)

roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nTest Metrics:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")

# Classification report
print(f"\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Normal', 'Fraud']
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticklabels(['Normal', 'Fraud'])
axes[0].set_yticklabels(['Normal', 'Fraud'])

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'evaluation_metrics.png'), dpi=150)
plt.show()

print(f"\n✓ Evaluation metrics plot saved to: {MODEL_SAVE_DIR}/evaluation_metrics.png")

## 9. Save Model and Preprocessors

In [ ]:
print("="*70)
print("Saving Model and Preprocessors")
print("="*70)

# Save final model
final_model_path = os.path.join(MODEL_SAVE_DIR, 'multimodal_fraud_detection_final.keras')
model.save(final_model_path)
print(f"\n✓ Final model saved to: {final_model_path}")

# Save preprocessors
preprocessor_path = os.path.join(MODEL_SAVE_DIR, 'preprocessor.pkl')
multimodal_preprocessor.save_preprocessors(preprocessor_path)
print(f"✓ Preprocessors saved to: {preprocessor_path}")

# Save model configuration
import json
config_path = os.path.join(MODEL_SAVE_DIR, 'model_config.json')
with open(config_path, 'w') as f:
    json.dump(MODEL_CONFIG, f, indent=2)
print(f"✓ Model configuration saved to: {config_path}")

print(f"\n{'='*70}")
print("Training Complete!")
print(f"{'='*70}")
print(f"\nAll model artifacts saved to: {MODEL_SAVE_DIR}")
print(f"\nTo view TensorBoard logs:")
print(f"  tensorboard --logdir={os.path.join(MODEL_SAVE_DIR, 'logs')}")
print(f"\nTo use the model for inference, load it with:")
print(f"  model = keras.models.load_model('{final_model_path}')")

## 10. Test Predictions on Sample Data

In [ ]:
print("="*70)
print("Testing Model on Sample Transactions")
print("="*70)

# Select a few test samples
n_samples = 5
sample_indices = np.random.choice(len(X_test_tabular), n_samples, replace=False)

for i, idx in enumerate(sample_indices):
    # Get sample data
    sample_tabular = X_test_tabular[idx:idx+1]
    sample_image = X_test_images[idx:idx+1]
    actual_label = y_test[idx][0]
    
    # Make prediction
    prediction_proba = model.predict([sample_tabular, sample_image], verbose=0)[0][0]
    prediction = 1 if prediction_proba > 0.5 else 0
    
    print(f"\nSample {i+1}:")
    print(f"  Actual: {'Fraud' if actual_label == 1 else 'Normal'}")
    print(f"  Predicted: {'Fraud' if prediction == 1 else 'Normal'}")
    print(f"  Confidence: {prediction_proba:.2%}")
    print(f"  Status: {'✓ CORRECT' if prediction == actual_label else '✗ INCORRECT'}")

print(f"\n{'='*70}")
print("✓ Prediction test complete!")